# Notebook 5. SmoothQuant

The three schemes so far quantize weights and leave activations in FP16. SmoothQuant (Xiao et al., ICML 2023) quantizes both to INT8, which is what an INT8 matmul kernel needs. The obstacle is that activations have outlier channels that weights do not, and SmoothQuant uses the scaling identity from Notebook 4 in the opposite direction: it divides the outlier channels of the activations by a factor and multiplies the matching weight columns by the same factor, so that both tensors become easy to quantize. This notebook shows the outlier problem, derives the smoothing factor, implements a W8A8 layer that matches Pare's, and runs SmoothQuant on Qwen2.5-0.5B.

Prerequisites are Notebooks 1 to 4.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, "..")   # import Pare from this checkout; `pip install pare-quant` also works
torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Activation outliers

In trained transformers a few hidden channels carry values tens of times larger than the rest, and they are the same channels for every token. The cell simulates 512 tokens of a 128-channel activation with three such channels.

In [ ]:
torch.manual_seed(1)
n_tokens, n_in = 512, 128
outliers = [8, 37, 92]
X = torch.randn(n_tokens, n_in) * 0.5
X[:, outliers] *= 30.0

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
im = axes[0].imshow(X[:40].numpy(), aspect="auto", cmap="RdBu_r", vmin=-20, vmax=20)
plt.colorbar(im, ax=axes[0])
axes[0].set_title("activations, 40 tokens by 128 channels")
axes[0].set_xlabel("channel")
axes[0].set_ylabel("token")
axes[1].bar(range(n_in), X.abs().amax(dim=0).numpy(), width=1)
axes[1].set_title("max |x| per channel")
axes[1].set_xlabel("channel")
plt.tight_layout()
plt.show()
print(f"typical channel std {X[:, 0].std():.2f}, outlier channel std {X[:, 8].std():.2f}")

## 2. Why INT8 activations fail without smoothing

An INT8 scale shared by the whole activation tensor is set by the outliers, so a typical channel uses only a few of the 256 codes. A scale per token still spans all channels of that token and has the same problem. A scale per channel would fix it, but it cannot be folded into the matmul: the per-channel factor multiplies the reduction dimension, so it has to be applied before the product, not after.

In [ ]:
def quantize_int8_sym(x, dim=None):
    # Symmetric INT8 with a scale per slice along `dim` (None: one scale for the tensor).
    scale = x.abs().amax(dim=dim, keepdim=dim is not None).clamp(min=1e-6) / 127.0
    q = torch.clamp(torch.round(x / scale), -128, 127)
    return q * scale


def mae(a, b):
    return (a - b).abs().mean().item()


print(f"per tensor:  MAE {mae(X, quantize_int8_sym(X)):.4f}")
print(f"per token:   MAE {mae(X, quantize_int8_sym(X, dim=1)):.4f}")
print(f"per channel: MAE {mae(X, quantize_int8_sym(X, dim=0)):.4f}   (not foldable)")

scale = X.abs().max() / 127.0
codes_used = torch.tensor([torch.round(X[:, j] / scale).unique().numel() for j in range(n_in)],
                          dtype=torch.float)
print(f"per-tensor codes in use: outlier channels about {codes_used[outliers].mean():.0f}, "
      f"others about {codes_used[[j for j in range(n_in) if j not in outliers]].mean():.0f}"
      " of 256")

## 3. Migrating the difficulty

For $Y = X W^T$ and a positive per-channel factor $s$,

$$Y = \underbrace{\left(X\,\mathrm{diag}(s)^{-1}\right)}_{\tilde{X}}\;\underbrace{\left(W\,\mathrm{diag}(s)\right)^T}_{\tilde{W}^T}.$$

SmoothQuant chooses $s_j$ so that the dynamic range of channel $j$ is shared between the two tensors,

$$s_j = \frac{\max_t \lvert X_{tj} \rvert^{\alpha}}{\max_i \lvert W_{ij} \rvert^{1-\alpha}},$$

where $\alpha$ is the migration strength. With $\alpha = 1$ every activation channel is scaled to the same maximum and the weights absorb all the variation; with $\alpha = 0$ the weights are equalized instead; $\alpha = 0.5$ gives both tensors the same per-channel maximum, $\sqrt{\max\lvert X_j \rvert \max\lvert W_j \rvert}$. The paper uses $0.5$ for most of the models it reports and a larger value for a model with heavier outliers. Pare's `QuantConfig.smooth_alpha` defaults to $0.5$, and `pare.schemes.smoothquant._smooth_factor` is the formula above with the maxima taken over all linear layers that share the input.

In [ ]:
from pare.schemes.smoothquant import _smooth_factor


def smooth_factor(x_max, W, alpha=0.5):
    w_max = W.abs().amax(dim=0).clamp(min=1e-6)       # per input channel, across output rows
    return (x_max.clamp(min=1e-6).pow(alpha) / w_max.pow(1 - alpha)).clamp(min=1e-6)


torch.manual_seed(2)
n_out = 64
W = torch.randn(n_out, n_in) * 0.05
x_max = X.abs().amax(dim=0)

linear = torch.nn.Linear(n_in, n_out, bias=False)
linear.weight.data = W.clone()
s = smooth_factor(x_max, W, alpha=0.5)
assert torch.allclose(s, _smooth_factor(x_max, [linear], 0.5))
print("smooth_factor matches pare.schemes.smoothquant._smooth_factor")

X_s, W_s = X / s, W * s
print(f"outlier channels: s = {[round(v, 2) for v in s[outliers].tolist()]}, "
      f"typical s = {s[0]:.2f}")
print(f"max |x| per channel before: {x_max[8]:.1f} (outlier) vs {x_max[0]:.1f}; after: "
      f"{X_s.abs().amax(dim=0)[8]:.2f} vs {X_s.abs().amax(dim=0)[0]:.2f}")
print(f"activation INT8 MAE (per token): before {mae(X, quantize_int8_sym(X, dim=1)):.4f}, "
      f"after {mae(X_s, quantize_int8_sym(X_s, dim=1)):.4f}")
print(f"weight INT8 MAE (per row):       before {mae(W, quantize_int8_sym(W, dim=1)):.5f}, "
      f"after {mae(W_s, quantize_int8_sym(W_s, dim=1)):.5f}")

The outlier channels are divided by a large factor and the activation error falls by more than an order of magnitude. The matching weight columns are multiplied by the same factor, become the largest entries of their rows, and widen the per-row step, so the weight error grows as well, by a smaller factor. Which side wins is decided by the output error, which the next section measures.

## 4. The migration strength

The two errors move in opposite directions as $\alpha$ grows. The cell sweeps $\alpha$ and measures the error of the W8A8 output on held-out tokens.

In [ ]:
X_test = torch.randn(200, n_in) * 0.5
X_test[:, outliers] *= 30.0
Y = X_test @ W.T

alphas = np.linspace(0, 1, 21)
act_err, w_err, out_err = [], [], []
for alpha in alphas:
    s_a = smooth_factor(x_max, W, alpha)
    X_q = quantize_int8_sym(X_test / s_a, dim=1)
    W_q = quantize_int8_sym(W * s_a, dim=1)
    act_err.append(mae(X_test / s_a, X_q) * s_a.mean().item())
    w_err.append(mae(W * s_a, W_q) / s_a.mean().item())
    out_err.append(((Y - X_q @ W_q.T).norm() / Y.norm()).item())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, values, title in zip(axes, (act_err, w_err, out_err),
                             ("activation error", "weight error", "relative output error")):
    ax.plot(alphas, values, "o-", markersize=4)
    ax.set_title(title)
    ax.set_xlabel("alpha")
    ax.set_xticks([0, 0.5, 1])
    ax.grid(True, alpha=0.4)
axes[2].axvline(0.5, color="gray", linestyle=":")
plt.tight_layout()
plt.show()
best = alphas[int(np.argmin(out_err))]
print(f"lowest output error at alpha = {best:.2f}; at alpha = 0.5 it is {out_err[10]:.4f}, "
      f"at alpha = 0 (no smoothing) {out_err[0]:.4f}")

## 5. A W8A8 layer, and where $1/s$ goes

At inference the weights are already $\tilde{W}$ and stored in INT8 with one scale per output row. The activations are quantized per token on the fly, with the scale $\max_j \lvert \tilde{x}_j \rvert / 127$ of each token, which costs one reduction per token and no calibration. $\mathrm{diag}(s)^{-1}$ is folded into the normalization layer that produces $X$, as in AWQ: for $\mathrm{LayerNorm}(x) = \gamma \odot \hat{x} + \beta$ the folded parameters are $\gamma / s$ and $\beta / s$, and the same holds for RMSNorm. Pare does this with the `_scale_ln_fcs` and `_scale_fc_fc` helpers from the AWQ module and the same fusion rules.

The layer below reproduces Pare's `QuantizedLinear` with `quantize_inputs=True`: asymmetric INT8 weights per row, symmetric INT8 activations per token, and the matmul in floating point on the dequantized values. The cell checks that it gives the same output as Pare's layer built from the same smoothed weights.

In [ ]:
from pare import QuantConfig
from pare.schemes.smoothquant import SmoothQuantQuantizer


class W8A8Linear(torch.nn.Module):
    def __init__(self, W_smooth, bias=None):
        super().__init__()
        w_min = W_smooth.amin(dim=1, keepdim=True).clamp(max=0.0)       # asymmetric INT8 per row
        w_max = W_smooth.amax(dim=1, keepdim=True).clamp(min=0.0)
        self.w_scale = ((w_max - w_min) / 255.0).clamp(min=1e-8)
        self.w_zero = torch.round(-128 - w_min / self.w_scale).clamp(-128, 127)
        self.w_q = torch.clamp(torch.round(W_smooth / self.w_scale + self.w_zero), -128, 127)
        self.bias = bias

    def forward(self, x_smooth):
        a_scale = x_smooth.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6) / 127.0
        x_q = torch.clamp(torch.round(x_smooth / a_scale), -128, 127) * a_scale
        W_hat = self.w_scale * (self.w_q - self.w_zero)
        return torch.nn.functional.linear(x_q, W_hat, self.bias)


# The notebook's layer against Pare's, both built from the smoothed weights W * s.
smoothed = torch.nn.Linear(n_in, n_out, bias=False)
smoothed.weight.data = (W * s).clone()
config = QuantConfig(bits=8, scheme="smoothquant", granularity="per_channel")
pare_layer = SmoothQuantQuantizer(config).quantize_layer(smoothed, "layer")
nb_layer = W8A8Linear(W * s)

x_in = X_test[:8] / s                                   # what the folded norm would emit
with torch.no_grad():
    assert torch.allclose(pare_layer(x_in), nb_layer(x_in), atol=1e-5)
    y_w8a8 = nb_layer(x_in)
print("W8A8Linear matches pare's QuantizedLinear with quantize_inputs=True")
y_ref = X_test[:8] @ W.T
print(f"relative output error of the W8A8 layer: {((y_ref - y_w8a8).norm() / y_ref.norm()):.4f}")

The matmul here runs on dequantized FP32 values. That is also what Pare does: its SmoothQuant path measures the accuracy of INT8 weights and activations and gives no speed gain. The speed gain of W8A8 comes from running the product on INT8 tensor cores, which needs a serving stack with INT8 GEMM kernels and is outside Pare.

## 6. SmoothQuant in Pare

As with AWQ, Pare's SmoothQuant runs block by block over `model.model.layers` and needs an `input_layernorm` to fold the scales into, so the cell uses Qwen2.5-0.5B. It compares FP32, RTN INT8 per channel (weights only) and SmoothQuant W8A8 with `smooth_alpha=0.5`, using 8 calibration sequences of 512 tokens and 20 evaluation windows of 512 tokens. `QuantConfig(calibration="percentile")` or `"mse"` replaces the per-channel maximum in $s_j$ by a clipped estimate; the README reports the three options on Qwen2.5-7B.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from pare import quantize
from pare.calibration.data import load_wikitext2_calibration
from pare.eval.perplexity import evaluate_perplexity

model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
calib = load_wikitext2_calibration(tokenizer, n_samples=8, seq_len=512)


def fresh_model():
    return AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)


def ppl(model):
    return evaluate_perplexity(
        model, tokenizer, dataset="wikitext2", seq_len=512, n_samples=20, device="cpu"
    )


results = {"FP32": ppl(fresh_model())}
configs = {
    "RTN INT8 weights": QuantConfig(bits=8, scheme="rtn", granularity="per_channel"),
    "SmoothQuant W8A8": QuantConfig(bits=8, scheme="smoothquant", granularity="per_channel"),
}
for label, cfg in configs.items():
    model = fresh_model()
    quantize(model, cfg, calibration_data=calib, device="cpu")
    results[label] = ppl(model)

for label, value in results.items():
    print(f"{label:>18}: PPL {value:6.2f}  ({value - results['FP32']:+.2f})")

INT8 weights alone cost almost nothing, as in Notebook 2. SmoothQuant adds INT8 activations on top, and the printed delta is the price of that on this model. On the 7B models in the README, SmoothQuant W8A8 adds 0.10 to 0.11 perplexity on Llama-2-7B, Llama-3-8B and Qwen2.5-7B.

## Exercises

1. In Section 4, set the outlier gain to 5 instead of 30 and rerun the sweep. Where does the best $\alpha$ move, and why?
2. Replace the per-token activation scale in `W8A8Linear` with a per-tensor scale fixed from the calibration data (`X_calib.abs().max() / 127`). Which setting of $\alpha$ now matters most, and how does the output error compare?
3. In Section 6, add `QuantConfig(bits=8, scheme="smoothquant", granularity="per_channel", calibration="percentile")` and compare its perplexity with the default.

## Summary

| Idea | Formula or setting | Role |
|------|--------------------|------|
| Outliers | a few channels, the same for every token | What makes INT8 activations hard |
| Smoothing | $\tilde{X} = X s^{-1}$, $\tilde{W} = W s$ | Moves range from activations to weights |
| Factor | $s_j = \max\lvert X_j \rvert^{\alpha} / \max\lvert W_j \rvert^{1-\alpha}$, $\alpha = 0.5$ | Shares the range between the two |
| Activations | symmetric INT8, one scale per token at run time | No activation calibration needed |
| Weights | asymmetric INT8, one scale per row | Same as RTN INT8 |
| Fusion | $1/s$ into the preceding norm | No extra operator |

Notebook 6 leaves uniform grids behind: NF4 places its sixteen levels where Gaussian weights are dense.